In [ ]:
print("hi")

In [ ]:
# ESPN: 2025 season projected fantasy points for players on your league's rosters
# Replace this with the number in your ESPN league URL.
LEAGUE_ID = 123456789
SEASON = 2025

# Leave these as None for a public league. For a private league, paste the
# values from your browser's ESPN_S2 and SWID cookies (do not share them).
ESPN_S2 = None
SWID = None

import pandas as pd
import requests

url = (f"https://fantasy.espn.com/apis/v3/games/ffl/seasons/{SEASON}"
       f"/segments/0/leagues/{LEAGUE_ID}")
cookies = {k: v for k, v in {"espn_s2": ESPN_S2, "SWID": SWID}.items() if v}
response = requests.get(url, params=[("view", "mRoster"), ("view", "mSettings")],
                        cookies=cookies, timeout=30)
response.raise_for_status()
league = response.json()

POSITION = {1: "QB", 2: "RB", 3: "WR", 4: "TE", 5: "K", 16: "D/ST"}
rows = []
for team in league["teams"]:
    team_name = team["location"] + " " + team["nickname"]
    for entry in team.get("roster", {}).get("entries", []):
        player = entry["playerPoolEntry"]["player"]
        # ESPN statSourceId 1 is its projection; appliedTotal already uses
        # this league's scoring rules. Prefer the full-season split (1).
        projections = [s for s in player.get("stats", []) if s.get("statSourceId") == 1]
        season_projection = next((s for s in projections if s.get("statSplitTypeId") == 1), None)
        if season_projection is None and projections:
            season_projection = projections[0]
        rows.append({
            "fantasy_team": team_name,
            "player": player["fullName"],
            "position": POSITION.get(player.get("defaultPositionId"), "Other"),
            "slot": entry.get("lineupSlotId"),
            "projected_points": None if season_projection is None else season_projection.get("appliedTotal"),
        })

projected_points = (pd.DataFrame(rows)
                    .dropna(subset=["projected_points"])
                    .sort_values("projected_points", ascending=False)
                    .reset_index(drop=True))
projected_points.index += 1
display(projected_points.style.format({"projected_points": "{:.1f}"}))

# Team-level 2025 projection totals:
display(projected_points.groupby("fantasy_team", as_index=False)["projected_points"]
       .sum().sort_values("projected_points", ascending=False)
       .style.format({"projected_points": "{:.1f}"}))


## Yahoo Historical Projection API Test

Tests Yahoo’s web-UI internal API anonymously before using any credentials. This is a small validation probe only: one league-settings request plus four 10-player weekly requests.


In [1]:
import json
import os
from collections.abc import Mapping

import pandas as pd
import requests
from IPython.display import display

SEASON = 2025
LEAGUE_ID = 707737
YAHOO_GAME_ID = 461  # Yahoo game key for the 2025 NFL season
LEAGUE_KEY = f"{YAHOO_GAME_ID}.l.{LEAGUE_ID}"
BASE_URL = "https://pub-api-ro.fantasysports.yahoo.com/fantasy/v2"
STAT_LABELS = None
HEADERS = {
    "User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 Chrome/131 Safari/537.36",
    "Accept": "application/json, text/plain, */*",
}


def walk_json(value):
    """Yield every dict/list member in a Yahoo JSON response, recursively."""
    yield value
    if isinstance(value, Mapping):
        for child in value.values():
            yield from walk_json(child)
    elif isinstance(value, list):
        for child in value:
            yield from walk_json(child)


def player_records(payload):
    """Find player records without relying on Yahoo's deeply nested array keys."""
    seen = set()
    for item in walk_json(payload):
        if isinstance(item, Mapping) and "player_id" in item and "name" in item:
            player_id = str(item["player_id"])
            if player_id not in seen:
                seen.add(player_id)
                yield item


def stat_label_map(league_key, session):
    """Fetch league stat-category labels so projected stat IDs are readable."""
    url = f"{BASE_URL}/league/{league_key}/settings"
    response = session.get(url, params={"format": "json_f"}, timeout=20)
    response.raise_for_status()
    labels = {}
    for item in walk_json(response.json()):
        if isinstance(item, Mapping) and {"stat_id", "display_name"} <= item.keys():
            labels[str(item["stat_id"])] = str(item["display_name"])
    return labels


def yahoo_projection_url(season, league_id, week, count=10):
    """The validated Yahoo web-UI endpoint for a historical weekly projection."""
    # Yahoo API game IDs are not calendar years (2025 NFL uses game ID 461).
    game_id = {2025: 461}.get(season, season)
    league_key = f"{game_id}.l.{league_id}"
    # sort=AR returns recognizable active/relevant players in the small sample.
    return (f"{BASE_URL}/league/{league_key}/players;start=0;count={count};sort=AR"
            f"/stats;type=week;week={week}")


def projection_stats(player, labels):
    stats = player.get("player_projected_stats", {}).get("stats", [])
    result = {}
    for item in stats:
        stat = item.get("stat", {})
        stat_id = str(stat.get("stat_id", "unknown"))
        label = labels.get(stat_id, f"stat_{stat_id}")
        result[f"projected_{label}"] = stat.get("value")
    return result


def get_yahoo_projections(season, league_id, week, *, count=10, session=None):
    """Return a small DataFrame of Yahoo historical weekly player projections.

    Uses no cookies by default. The endpoint includes Yahoo's projected total
    and its individual projected scoring-stat categories when they are present.
    """
    session = session or requests.Session()
    session.headers.update(HEADERS)
    game_id = {2025: 461}.get(season, season)
    league_key = f"{game_id}.l.{league_id}"
    url = yahoo_projection_url(season, league_id, week, count)
    params = {
        "format": "json_f",
        "show_projected_stats": "1",
        "show_live_projected_points": "1",
    }
    response = session.get(url, params=params, timeout=20)
    body_preview = response.text[:800]
    print(f"URL requested: {response.url}")
    print(f"HTTP status: {response.status_code}")
    print(f"Authentication required: {'yes' if response.status_code in (401, 403) or 'Must login first' in response.text else 'no'}")
    if not response.ok:
        print("Response body (first 800 characters):\n", body_preview)
        response.raise_for_status()
    payload = response.json()
    global STAT_LABELS
    if STAT_LABELS is None:
        STAT_LABELS = stat_label_map(league_key, session)
    labels = STAT_LABELS
    rows = []
    for player in player_records(payload):
        projected = player.get("player_projected_points", {})
        row = {
            "season": season,
            "week": int(projected.get("week", week)),
            "league_id": league_id,
            "player_id": str(player["player_id"]),
            "player_name": player.get("name", {}).get("full"),
            "nfl_team": player.get("editorial_team_abbr"),
            "position": player.get("display_position"),
            "projected_points": pd.to_numeric(projected.get("total"), errors="coerce"),
        }
        row.update(projection_stats(player, labels))
        rows.append(row)
    df = pd.DataFrame(rows)
    print(f"Player records returned: {len(df)}")
    if not df.empty:
        print("Projection week(s) returned:", sorted(df["week"].unique().tolist()))
    return df


In [2]:
# Anonymous validation: only four weekly player requests, plus settings metadata.
# The DataFrame samples include Yahoo's projected_points and projected-stat columns.
weeks_to_test = [1, 5, 10, 17]
anonymous_session = requests.Session()
anonymous_session.headers.update(HEADERS)
weekly_projections = {}

for week in weeks_to_test:
    print(f"\n--- Week {week} ---")
    weekly_projections[week] = get_yahoo_projections(
        SEASON, LEAGUE_ID, week, count=10, session=anonymous_session
    )

print("\nWeek 5 sample (first 10 returned players):")
display(weekly_projections[5].head(10))

# Show one recognizable player present in every requested week. Josh Allen is
# preferred when Yahoo returns him; otherwise choose the first common player.
common_ids = set.intersection(*(set(df["player_id"]) for df in weekly_projections.values()))
preferred = ["Josh Allen"]
comparison_id = next((
    player_id for player_id in common_ids
    if any((df.loc[df["player_id"] == player_id, "player_name"] == name).any()
           for name in preferred for df in weekly_projections.values())
), next(iter(common_ids), None))

if comparison_id:
    comparison = pd.concat(weekly_projections.values(), ignore_index=True)
    comparison = comparison.loc[comparison["player_id"] == comparison_id,
                                ["week", "player_id", "player_name", "nfl_team", "position", "projected_points"]]
    print("Historical weekly projection comparison:")
    display(comparison.sort_values("week").reset_index(drop=True))
    print("Projected points differ by week:", comparison["projected_points"].nunique() > 1)
else:
    print("No player appeared in all four small samples; increase count slightly if needed.")



--- Week 1 ---


URL requested: https://pub-api-ro.fantasysports.yahoo.com/fantasy/v2/league/461.l.707737/players;start=0;count=10;sort=AR/stats;type=week;week=1?format=json_f&show_projected_stats=1&show_live_projected_points=1
HTTP status: 200
Authentication required: no
Player records returned: 10
Projection week(s) returned: [1]

--- Week 5 ---


URL requested: https://pub-api-ro.fantasysports.yahoo.com/fantasy/v2/league/461.l.707737/players;start=0;count=10;sort=AR/stats;type=week;week=5?format=json_f&show_projected_stats=1&show_live_projected_points=1
HTTP status: 200
Authentication required: no
Player records returned: 10
Projection week(s) returned: [5]

--- Week 10 ---


URL requested: https://pub-api-ro.fantasysports.yahoo.com/fantasy/v2/league/461.l.707737/players;start=0;count=10;sort=AR/stats;type=week;week=10?format=json_f&show_projected_stats=1&show_live_projected_points=1
HTTP status: 200
Authentication required: no
Player records returned: 10
Projection week(s) returned: [10]

--- Week 17 ---


URL requested: https://pub-api-ro.fantasysports.yahoo.com/fantasy/v2/league/461.l.707737/players;start=0;count=10;sort=AR/stats;type=week;week=17?format=json_f&show_projected_stats=1&show_live_projected_points=1
HTTP status: 200
Authentication required: no
Player records returned: 10
Projection week(s) returned: [17]

Week 5 sample (first 10 returned players):


,season,week,league_id,player_id,player_name,nfl_team,position,projected_points,projected_Pass Yds,projected_Pass TD,...,projected_Rush Yds,projected_Rush TD,projected_Targets,projected_Rec,projected_Rec Yds,projected_Rec TD,projected_Ret TD,projected_2-PT,projected_Fum Lost,projected_Fum Ret TD
0,2025,5,707737,30977,Josh Allen,Buf,QB,21.14,239,1.6,...,32.1,0.5,0,0,0,0,0,0.1,0.1,0
1,2025,5,707737,30121,Christian McCaffrey,SF,RB,22.03,0,0,...,67.6,0.5,8.2,6.1,48.5,0.3,0,0.0,0.1,0
2,2025,5,707737,40881,Drake Maye,NE,QB,16.98,217,1.2,...,31.4,0.2,0,0,0,0,0,0.1,0.1,0
3,2025,5,707737,9265,Matthew Stafford,LAR,QB,16.15,249,1.5,...,5.4,0.1,0,0,0,0,0,0.1,0.1,0
4,2025,5,707737,33389,Trevor Lawrence,Jax,QB,15.86,236,1.4,...,12.5,0.1,0,0,0,0,0,0.1,0.1,0
5,2025,5,707737,32711,Jonathan Taylor,Ind,RB,17.94,0,0,...,89.5,0.7,3.0,2.4,19.0,0.1,0,0.0,0.1,0
6,2025,5,707737,40055,Bijan Robinson,Atl,RB,0.00,0,0,...,0,0,0,0,0,0,0,0,0,0
7,2025,5,707737,40059,Jahmyr Gibbs,Det,RB,17.82,0,0,...,66.1,0.7,4.2,3.5,26.9,0.2,0,0.0,0.1,0
8,2025,5,707737,40900,Caleb Williams,Chi,QB,0.00,0,0,...,0,0,0,0,0,0,0,0,0,0
9,2025,5,707737,29369,Dak Prescott,Dal,QB,16.59,252,1.5,...,8.3,0.1,0,0,0,0,0,0.1,0.1,0


Historical weekly projection comparison:


,week,player_id,player_name,nfl_team,position,projected_points
0,1,30977,Josh Allen,Buf,QB,20.81
1,5,30977,Josh Allen,Buf,QB,21.14
2,10,30977,Josh Allen,Buf,QB,21.97
3,17,30977,Josh Allen,Buf,QB,19.70


Projected points differ by week: True


In [3]:
# OPTIONAL AUTHENTICATION (do not run unless anonymous access later stops working).
# Keep secrets outside the notebook: set Yahoo cookies as environment variables
# in your shell/kernel environment, then pass this session to get_yahoo_projections.
# This test intentionally does NOT hard-code or request any credentials.
YAHOO_COOKIE_A = os.environ.get("YAHOO_COOKIE_A")
YAHOO_COOKIE_B = os.environ.get("YAHOO_COOKIE_B")
optional_authenticated_session = requests.Session()
optional_authenticated_session.headers.update(HEADERS)
optional_authenticated_session.cookies.update({
    key: value for key, value in {
        "COOKIE_NAME_A": YAHOO_COOKIE_A,  # replace only the cookie *name* locally
        "COOKIE_NAME_B": YAHOO_COOKIE_B,
    }.items() if value
})
print("Optional authentication session prepared; no request was sent.")


Optional authentication session prepared; no request was sent.


### Result summary

The validated internal endpoint is:

```text
https://pub-api-ro.fantasysports.yahoo.com/fantasy/v2/league/461.l.707737/players;start=0;count=10;sort=AR/stats;type=week;week={WEEK}?format=json_f&show_projected_stats=1&show_live_projected_points=1
```

It returned HTTP 200 JSON without Yahoo cookies for this public league and includes `player_projected_points.total` plus `player_projected_stats.stats` (the individual scoring-stat projections). The validation cell requests completed Weeks 1, 5, 10, and 17 and prints one shared player's values by week, explicitly testing that the projections are historical weekly values rather than a repeated current value.

This is an internal web-UI API rather than Yahoo's documented developer API. It is suitable for a later, rate-limited backfill experiment only after the four-week comparison remains successful; its schema and anonymous-access behavior are not guaranteed by Yahoo. No full historical download is performed here.
